<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/ML/W10/Q2/ML_ASSIGNMENT_W_10_Q_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# -------IMPLEMENT L1 AND L2 REGULARIZATION PENALTIES---------
# ============================================================

import os
import sys
import subprocess
import zipfile

# ============================================================
# CREATE PYTHON SOURCE CODE
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

In [5]:
# ============================================================
# 1. SAMPLE DATASET
# ============================================================

X = np.array([
    [0.5, 1.2, 0.05],
    [1.0, 2.4, 0.12],
    [1.5, 3.1, 0.88],
    [2.0, 4.5, 0.22]
])

y = np.array([10.2, 20.5, 29.8, 41.0])

feature_names = ['X1', 'X2', 'X3']



In [6]:
# ============================================================
# 2. STANDARDIZE FEATURES
# ============================================================

X_mean = X.mean(axis=0)
X_std = X.std(axis=0)

Z = (X - X_mean) / X_std

y_mean = y.mean()
yc = y - y_mean

In [7]:
# ============================================================
# 3. L1 REGULARIZATION (LASSO)
# ============================================================

def l1_regularization(X, y, lam, iterations=10000):

    n, p = X.shape
    weights = np.zeros(p)

    # Coordinate descent optimization
    for _ in range(iterations):

        old_weights = weights.copy()

        for j in range(p):

            residual = y - X @ weights + X[:, j] * weights[j]

            rho = np.mean(X[:, j] * residual)

            # Soft thresholding
            if rho > lam:
                weights[j] = rho - lam

            elif rho < -lam:
                weights[j] = rho + lam

            else:
                weights[j] = 0.0

        if np.max(np.abs(weights - old_weights)) < 1e-12:
            break

    return weights

In [ ]:
# ============================================================
# 4. L2 REGULARIZATION (RIDGE)
# ============================================================

def l2_regularization(X, y, lam):

    n = X.shape[0]

    # Ridge closed-form solution
    identity = np.eye(X.shape[1])

    weights = np.linalg.solve(
        (X.T @ X) / n + 2 * lam * identity,
        (X.T @ y) / n
    )

    return weights

In [10]:
# ============================================================
# 5. REGULARIZATION LOSS FUNCTIONS
# ============================================================

def l1_loss(y_true, y_pred, weights, lam):

    mse_loss = np.mean((y_true - y_pred) ** 2) / 2

    penalty = lam * np.sum(np.abs(weights))

    return mse_loss + penalty


def l2_loss(y_true, y_pred, weights, lam):

    mse_loss = np.mean((y_true - y_pred) ** 2) / 2

    penalty = lam * np.sum(weights ** 2)

    return mse_loss + penalty

In [11]:
# ============================================================
# 6. TEST DIFFERENT LAMBDA VALUES
# ============================================================

lambda_values = np.linspace(0.1, 10, 20)

results = []

for lam in lambda_values:

    # Calculate L1 and L2 weights
    w_l1 = l1_regularization(Z, yc, lam)

    w_l2 = l2_regularization(Z, yc, lam)

    # Convert standardized coefficients to original scale
    original_l1 = w_l1 / X_std
    original_l2 = w_l2 / X_std

    # Calculate intercepts
    intercept_l1 = y_mean - np.dot(X_mean, original_l1)

    intercept_l2 = y_mean - np.dot(X_mean, original_l2)

    # Store results
    results.append({
        'Lambda': lam,

        'L1_Weight_X1': original_l1[0],
        'L1_Weight_X2': original_l1[1],
        'L1_Weight_X3': original_l1[2],

        'L2_Weight_X1': original_l2[0],
        'L2_Weight_X2': original_l2[1],
        'L2_Weight_X3': original_l2[2],

        'L1_Intercept': intercept_l1,
        'L2_Intercept': intercept_l2
    })